In [1]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch

from openretina.data_io.hoefling_2024.stimuli import movies_from_pickle
from openretina.utils.plotting import (
    numpy_to_mp4_video,
)
from openretina.utils.file_utils import get_local_file_path
from openretina.utils.h5_handling import load_h5_into_dict
from openretina.data_io.cyclers import LongCycler, ShortCycler
from openretina.data_io.hoefling_2024.dataloaders import natmov_dataloaders_v2
from openretina.data_io.hoefling_2024.responses import filter_responses, make_final_responses
from openretina.data_io.hoefling_2024.stimuli import movies_from_pickle
import os
import hydra

In [2]:
with hydra.initialize(config_path=os.path.join("..", "configs"), version_base="1.3"):
    cfg = hydra.compose(config_name="hoefling_2024_core_readout_low_res.yaml")

/home/bethge/bkr618/openretina_cache/notebook_example/tensorboard

In [3]:
your_chosen_root_folder = "/home/bethge/bkr618/openretina_cache"  # Change this with your desired path.

cfg.paths.cache_dir = your_chosen_root_folder

# We will also overwrite the output directory for the logs/model to the local folder.
cfg.paths.log_dir = your_chosen_root_folder
cfg.paths.output_dir = your_chosen_root_folder

os.environ["OPENRETINA_CACHE_DIRECTORY"] = your_chosen_root_folder

In [4]:
file_path = '/home/bethge/bkr618/openretina_cache/euler_lab/hoefling_2024/stimuli/rgc_natstim_72x64_joint_normalized_2024-10-11.pkl'
movie_stimuli = movies_from_pickle(file_path)

In [5]:
responses_path = "/home/bethge/bkr618/openretina_cache/data/euler_lab/hoefling_2024/responses/rgc_natstim_2024-08-14.h5"
responses_dict = load_h5_into_dict(file_path=responses_path)

filtered_responses_dict = filter_responses(responses_dict, **cfg.quality_checks)

final_responses = make_final_responses(filtered_responses_dict, response_type="natural")

Loading HDF5 file contents:   0%|          | 0/2077 [00:00<?, ?item/s]

Original dataset contains 7863 neurons over 67 fields
 ------------------------------------ 
Dropped 0 fields that did not contain the target cell types (67 remaining)
Overall, dropped 3034 neurons of non-target cell types (-38.59%).
 ------------------------------------ 
Dropped 0 fields with quality indices below threshold (67 remaining)
Overall, dropped 980 neurons over quality checks (-20.29%).
 ------------------------------------ 
Dropped 0 fields with classifier confidences below 0.25
Overall, dropped 705 neurons with classifier confidences below 0.25 (-18.32%).
 ------------------------------------ 
 ------------------------------------ 
Final dataset contains 3144 neurons over 67 fields
Total number of cells dropped: 4719 (-60.02%)


Upsampling natural spikes traces to get final responses.:   0%|          | 0/67 [00:00<?, ?it/s]

In [6]:
dataloaders = natmov_dataloaders_v2(
    neuron_data_dictionary=final_responses,
    movies_dictionary=movie_stimuli,
    allow_over_boundaries=True,
    batch_size=128,
    train_chunk_size=50,
    validation_clip_indices=cfg.dataloader.validation_clip_indices,
)

Creating movie dataloaders:   0%|          | 0/67 [00:00<?, ?it/s]

In [7]:
from openretina.data_io.base import compute_data_info
data_info = compute_data_info(neuron_data_dictionary=final_responses, movies_dictionary=movie_stimuli)


In [8]:
train_loader = LongCycler(dataloaders["train"])
val_loader = ShortCycler(dataloaders["validation"])

In [9]:
n_neurons_dict = data_info["n_neurons_dict"]
from openretina.modules.core.transformer_core import ViViTCore
from openretina.modules.readout.multi_readout import MultiSampledGaussianReadout
from openretina.models.core_readout import UnifiedCoreReadout


vivit_core_cfg = {
    "_target_": "openretina.modules.core.transformer_core.ViViTCore",
    "in_shape": (2, 50, 72, 64),
    "Demb": 64,
    "patch_size": 12,
    "temporal_patch_size": 20,
    "num_spatial_blocks": 15,
    "num_temporal_blocks": 4,
    "num_heads": 4,
    "mlp_ratio": 4.0,
    "dropout": 0.2,
    "pad_frame": False,
    "temporal_stride": 12,
    "spatial_stride": 6,
    "ptoken": 0.0,
    "norm": "rmsnorm",
    "patch_mode": True,
    "pos_encoding": 5,
    "reg_tokens": 10,
    "ff_activation": "relu",
    "mha_dropout": 0.05,
    "drop_path": 0.2,
    "ff_dropout": 0.2,
    "use_causal_attention": True,
    "smooth_lambda": 0.2,
    "reg_scale": 0.1
}

readout_cfg = {
    "_target_": "openretina.modules.readout.multi_readout.MultiSampledGaussianReadout",
    "bias": True,
    "n_neurons_dict": n_neurons_dict,
    "init_mu_range": 0.1,
    "init_sigma_range": 0.15,
    "gamma": 0.4,
    "reg_avg": False,
}

model = UnifiedCoreReadout(
    in_shape=(2,50,72,64),
    core=vivit_core_cfg,
    readout=readout_cfg,
    hidden_channels=[],
    n_neurons_dict=n_neurons_dict
)

model = model.to('cuda')

InstantiationException: Error in call to target 'openretina.modules.readout.multi_readout.MultiSampledGaussianReadout':
TypeError("MultiSampledGaussianReadout.__init__() missing 1 required positional argument: 'kernel_size'")

In [ ]:
import lightning

In [ ]:
log_save_path = os.path.join(cfg.paths.output_dir, "notebook_example")
os.makedirs(log_save_path, exist_ok=True)

logger = lightning.pytorch.loggers.TensorBoardLogger(
    name="tensorboard/",
    save_dir=log_save_path,
)

In [13]:
early_stopping = lightning.pytorch.callbacks.EarlyStopping(
    monitor="val_correlation",
    patience=20,
    mode="max",
    verbose=False,
    min_delta=0.001,
)

lr_monitor = lightning.pytorch.callbacks.LearningRateMonitor(logging_interval="epoch")

model_checkpoint = lightning.pytorch.callbacks.ModelCheckpoint(
    monitor="val_correlation", mode="max", save_weights_only=False
)

In [14]:
trainer = lightning.Trainer(max_epochs=100, logger=logger, callbacks=[early_stopping, lr_monitor, model_checkpoint], accumulate_grad_batches=5, precision = '16-mixed') #add precision

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [15]:
trainer.fit(model, train_loader, val_loader)

You are using a CUDA device ('NVIDIA A100-PCIE-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/bethge/bkr618/open-retina/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:231: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name            | Type                        | Params | Mode 
------------------------------------------------------------------------
0 | core            | ViViTCore                   | 1.3 M  | train
1 | readout         | MultiSampledGaussianReadout | 223 K  | train
2 | loss            | PoissonLoss3d               | 0      

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [16]:
test_loader = ShortCycler(dataloaders["test"])

In [17]:

trainer.test(model, dataloaders=[train_loader, val_loader, test_loader], ckpt_path="best")

Restoring states from the checkpoint path at /home/bethge/bkr618/openretina_cache/notebook_example/tensorboard/version_71/checkpoints/epoch=6-step=189.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/bethge/bkr618/openretina_cache/notebook_example/tensorboard/version_71/checkpoints/epoch=6-step=189.ckpt


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃       DataLoader 1        ┃       DataLoader 2        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│     test_correlation      │    0.08063589036464691    │    0.12057337164878845    │    0.2681322395801544     │
│         test_loss         │     79.3514175415039      │     82.62376403808594     │     20.81061363220215     │
└───────────────────────────┴───────────────────────────┴───────────────────────────┴───────────────────────────┘

[{'test_loss/dataloader_idx_0': 79.3514175415039,
  'test_correlation/dataloader_idx_0': 0.08063589036464691},
 {'test_loss/dataloader_idx_1': 82.62376403808594,
  'test_correlation/dataloader_idx_1': 0.12057337164878845},
 {'test_loss/dataloader_idx_2': 20.81061363220215,
  'test_correlation/dataloader_idx_2': 0.2681322395801544}]

In [9]:
# First, put the stimuli in a torch tensor, which is what the model expects.
stim = torch.Tensor(movie_stimuli.test_movie).to(model.device)

# Second, we need to select one of the many experimental sessions the model was trained on to visualize a response.
example_session = model.readout.sessions[0]  # Can pick any number as long as it is in range

with torch.no_grad():
    predicted_response = model.forward(stim.unsqueeze(0), data_key=example_session)
predicted_response_numpy = predicted_response.squeeze().cpu().numpy()

In [10]:
import ipywidgets as widgets
# Create a dropdown for neuron selection
neuron_selector = widgets.Dropdown(
    options=list(range(predicted_response_numpy.shape[1])),
    value=0,
    description="Neuron:",
)


# Define the plotting function
def plot_response(neuron_idx):
    plt.figure(figsize=(12, 6))
    plt.plot(predicted_response_numpy[:, neuron_idx])
    plt.xlabel("Time [frames]")
    plt.ylabel("Response [a.u.]")
    sns.despine()
    plt.show()


# Create an interactive widget
widgets.interactive(plot_response, neuron_idx=neuron_selector)

interactive(children=(Dropdown(description='Neuron:', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 1…